In [ ]:
from train_neural_network_with_pytorch import loss_function, predicted_labels, mnist_transform
!pip install torch
!pip install torchvision
!pip install tqdm
!pip install matplotlib
!pip install --upgrade "numpy<2"

In [1]:
import torch
import torchvision
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import Compose, ToTensor, Normalize, Lambda

print("Torch version: ", torch.__version__)
print("Torchvision version: ", torchvision.__version__)

Torch version:  2.11.0+cu130
Torchvision version:  0.26.0+cu130


In [2]:
def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def move_to_device(tensor: torch.Tensor, device: torch.device) -> torch.Tensor:
    return tensor.to(device, non_blocking=True)

device = get_device()
print("Device: ", device)

Device:  cpu


In [3]:
model = nn.Sequential(
    nn.Linear(28*28, 100),
    nn.ReLU(),
    nn.Linear(500, 300),
    nn.ReLU(),
    nn.Linear(300, 10)
)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def forward(inputs: torch.Tensor) -> torch.Tensor:
    return model(inputs)

def predict(input_images: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        logits = forward(input_images)
        predicted_labels = torch.argmax(logits, dim=-1)
        return predicted_labels


def train_on_batch(batch: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    logits = forward(batch)
    loss = loss_function(logits, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
mnist_transform = Compose([ToTensor(), Lambda(lambda image: image.view(28*28))])